In [1]:
spark.stop()

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("iceberg-lab")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "hive")
    .config("spark.sql.catalog.lakehouse.uri", "thrift://metastore:9083")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3a://warehouse/")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

26/03/11 18:07:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
spark.sql("SHOW NAMESPACES IN lakehouse").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|demo     |
+---------+



In [4]:
spark.sql("""
create table if not exists lakehouse.demo.users(
id bigint, name string, created_at timestamp)
using iceberg
""")

26/03/11 18:10:25 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[]

In [5]:
spark.sql("SHOW TABLES IN lakehouse.demo").show(truncate=False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|demo     |users    |false      |
+---------+---------+-----------+



In [6]:
spark.sql("""
INSERT INTO lakehouse.demo.users VALUES
(1, 'An', current_timestamp()),
(2, 'Binh', current_timestamp())
""")

26/03/11 18:11:14 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to demo.db/users/data/00000-0-4399318e-de07-4338-a20a-a912f2f32457-0-00001.parquet. This is unsupported
                                                                                

DataFrame[]

In [7]:
spark.sql("SELECT * FROM lakehouse.demo.users").show(truncate=False)

+---+----+--------------------------+
|id |name|created_at                |
+---+----+--------------------------+
|1  |An  |2026-03-11 18:11:14.024482|
|2  |Binh|2026-03-11 18:11:14.024482|
+---+----+--------------------------+

